#Importing the Libraries

In [4]:
import pandas as pd
import numpy as np
import re
import string
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

#Load Datasets

In [5]:
df=pd.read_csv("cyberbullying_tweets.csv")
print("Original Labels:")
print(df['cyberbullying_type'].value_counts())

Original Labels:
cyberbullying_type
religion               7998
age                    7992
gender                 7973
ethnicity              7961
not_cyberbullying      7945
other_cyberbullying    7823
Name: count, dtype: int64


#Applying Binary-Values

In [6]:
df['label']=df['cyberbullying_type'].apply(lambda x: 0 if x == 'not_cyberbullying' else 1)
print("\nBinary Labels:")
print(df['label'].value_counts())


Binary Labels:
label
1    39747
0     7945
Name: count, dtype: int64


#Clean the text

In [7]:
stop_words=set(stopwords.words('english'))
def clean_text(text):
    text=str(text).lower()
    text=re.sub(r"http\S+", "",text)
    text=re.sub(r"@\w+", "",text)
    text=re.sub(r"#", "",text)
    text=re.sub(r"[^\w\s]", "",text)
    text=" ".join([w for w in text.split() if w not in stop_words])
    return text

df['clean_text']=df['tweet_text'].apply(clean_text)

#Train_Test_Split the data

In [8]:
x=df['clean_text']
y=df['label']
x_train,x_test,y_train,y_test=train_test_split(x, y,test_size=0.2,stratify=y,random_state=42)


Vectorization

In [9]:
vectorizer=TfidfVectorizer(max_features=10000,ngram_range=(1,2))
x_train_vec=vectorizer.fit_transform(x_train)
x_test_vec=vectorizer.transform(x_test)

#Logistic Regression



In [10]:
lr=LogisticRegression(max_iter=1000,class_weight='balanced')
lr.fit(x_train_vec, y_train)
lr_pred=lr.predict(x_test_vec)
print("\nLogistic Regression")
print("Accuracy:",accuracy_score(y_test,lr_pred))
print(classification_report(y_test,lr_pred))


Logistic Regression
Accuracy: 0.8042771779012475
              precision    recall  f1-score   support

           0       0.45      0.87      0.60      1589
           1       0.97      0.79      0.87      7950

    accuracy                           0.80      9539
   macro avg       0.71      0.83      0.73      9539
weighted avg       0.88      0.80      0.83      9539



##Naive Bayes

In [11]:
nb=MultinomialNB()
nb.fit(x_train_vec,y_train)
nb_pred=nb.predict(x_test_vec)
print("\nNaive Bayes")
print("Accuracy:",accuracy_score(y_test,nb_pred))
print(classification_report(y_test,nb_pred))


Naive Bayes
Accuracy: 0.8596288919173918
              precision    recall  f1-score   support

           0       0.74      0.24      0.37      1589
           1       0.87      0.98      0.92      7950

    accuracy                           0.86      9539
   macro avg       0.80      0.61      0.64      9539
weighted avg       0.85      0.86      0.83      9539



In [13]:
svm=LinearSVC(class_weight='balanced')
svm.fit(x_train_vec,y_train)
svm_pred = svm.predict(x_test_vec)
print("\n--- SVM ---")
print("Accuracy:", accuracy_score(y_test, svm_pred))
print(classification_report(y_test, svm_pred))


--- SVM ---
Accuracy: 0.8081559911940455
              precision    recall  f1-score   support

           0       0.46      0.77      0.57      1589
           1       0.95      0.82      0.88      7950

    accuracy                           0.81      9539
   macro avg       0.70      0.79      0.72      9539
weighted avg       0.86      0.81      0.83      9539



In [30]:
print("\nFINAL COMPARISON")
print("Logistic Regression:",accuracy_score(y_test,lr_pred))
print("SVM:",accuracy_score(y_test,svm_pred))
print("Naive Bayes:",accuracy_score(y_test,nb_pred))


FINAL COMPARISON
Logistic Regression: 0.8042771779012475
SVM: 0.8081559911940455
Naive Bayes: 0.8596288919173918


Although Naive Bayes achieved higher accuracy, it performed poorly on the minority class. SVM provided balanced performance across both classes, so it is the best model for this problem.

#SAVE THE MODEL

In [34]:
joblib.dump(svm,"cyberbullying_model.pkl")
joblib.dump(vectorizer,"vectorizer.pkl")
print("SVM  model saved successfully!")

SVM  model saved successfully!


In [35]:
model=joblib.load("cyberbullying_model.pkl")
vectorizer=joblib.load("vectorizer.pkl")
def predict(text):
    text=clean_text(text)
    vec=vectorizer.transform([text])
    pred=model.predict(vec)[0]

    return "Cyberbullying" if pred == 1 else "Not Cyberbullying"

In [36]:
print(predict("you are stupid"))
print(predict("good morning friend"))
print(predict("you are a idiot"))
print(predict("you are a good guy"))
print(predict("you are a bad guy"))
print(predict("you are a bad girl"))
print(predict("you are a idiot"))

Cyberbullying
Not Cyberbullying
Cyberbullying
Not Cyberbullying
Not Cyberbullying
Cyberbullying
Cyberbullying
